<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/SVM(70_30).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ML/skin_recommendation_dataset.csv")

In [3]:
import re

def clean_text(x):
    # convert to string
    x = str(x)

    # remove everything except letters, numbers, space, dash, comma
    x = re.sub(r"[^a-zA-Z0-9,\-\s]", "", x)

    # split
    items = x.split(",")

    # clean each word
    items = [i.strip().lower() for i in items if i.strip() != ""]

    return items

In [4]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [5]:
data = df[['skintype','skin_condition','notable_effects','product_type']].copy()

In [6]:
data['skintype'] = data['skintype'].apply(clean_text)
data['skin_condition'] = data['skin_condition'].apply(clean_text)
data['notable_effects'] = data['notable_effects'].apply(clean_text)

In [7]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_skin = MultiLabelBinarizer()
mlb_condition = MultiLabelBinarizer()
mlb_effects = MultiLabelBinarizer()

skin_features = pd.DataFrame(
    mlb_skin.fit_transform(data['skintype']),
    columns=mlb_skin.classes_
)

condition_features = pd.DataFrame(
    mlb_condition.fit_transform(data['skin_condition']),
    columns=mlb_condition.classes_
)

effects_features = pd.DataFrame(
    mlb_effects.fit_transform(data['notable_effects']),
    columns=mlb_effects.classes_
)

X = pd.concat([skin_features, condition_features, effects_features], axis=1)

In [8]:
X = pd.concat([
    skin_features,
    condition_features,
    effects_features
], axis=1)

In [9]:
le = LabelEncoder()
y = le.fit_transform(data['product_type'])

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
svm = SVC(
    kernel='rbf',   # best for your dataset
    C=1,
    gamma='scale'
)

svm.fit(X_train_scaled, y_train)

SVC(C=1)

In [13]:
y_pred = svm.predict(X_test_scaled)

In [14]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.4945652173913043


In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.54      0.41      0.47        61
           1       0.35      0.36      0.36        74
           2       0.50      0.68      0.58        92
           3       0.78      0.67      0.72        64
           4       0.38      0.31      0.34        77

    accuracy                           0.49       368
   macro avg       0.51      0.49      0.49       368
weighted avg       0.50      0.49      0.49       368



In [16]:
f1 = f1_score(y_test, y_pred, average='weighted')

print("Weighted F1 Score:", f1)

Weighted F1 Score: 0.4909738266790745


In [17]:
import pandas as pd
import numpy as np

# ==============================
# 1. USER INPUT (NUMBER BASED)
# ==============================

print("\nSelect Skin Type:")
for i, val in enumerate(mlb_skin.classes_):
    print(i, ":", val)

skin_choice = int(input("Enter number: "))
user_skin = [mlb_skin.classes_[skin_choice]]


print("\nSelect Skin Condition:")
for i, val in enumerate(mlb_condition.classes_):
    print(i, ":", val)

cond_choice = int(input("Enter number: "))
user_condition = [mlb_condition.classes_[cond_choice]]


print("\nSelect Effect:")
for i, val in enumerate(mlb_effects.classes_):
    print(i, ":", val)

effect_choice = int(input("Enter number: "))
user_effect = [mlb_effects.classes_[effect_choice]]


# ==============================
# 2. CREATE INPUT VECTOR
# ==============================

user_df = pd.DataFrame(columns=X.columns)
user_df.loc[0] = 0

for val in user_skin + user_condition + user_effect:
    if val in user_df.columns:
        user_df.loc[0, val] = 1


# ==============================
# 3. SCALE INPUT
# ==============================

user_input_scaled = scaler.transform(user_df)


# ==============================
# 4. SVM PREDICTION
# ==============================

prediction = svm.predict(user_input_scaled)
predicted_type = le.inverse_transform(prediction)

print("\n✅ Predicted Product Type:", predicted_type[0])


# ==============================
# 5. SMART RECOMMENDATION SYSTEM
# ==============================

filtered = df[df['product_type'] == predicted_type[0]].copy()

# clean text columns
filtered['notable_effects'] = filtered['notable_effects'].astype(str).str.lower()
filtered['skin_condition'] = filtered['skin_condition'].astype(str).str.lower()
filtered['skintype'] = filtered['skintype'].astype(str).str.lower()


# scoring function
def calculate_score(row):
    score = 0

    # effect match (highest priority)
    if user_effect[0] in row['notable_effects']:
        score += 3

    # condition match
    if user_condition[0] in row['skin_condition']:
        score += 2

    # skin type match
    if user_skin[0] in row['skintype']:
        score += 1

    return score


# apply scoring
filtered['score'] = filtered.apply(calculate_score, axis=1)

# sort best matches
filtered = filtered.sort_values(by='score', ascending=False)


# ==============================
# 6. SHOW TOP 3 RESULTS
# ==============================

print("\n🔥 Top Recommended Products:\n")

top_results = filtered.head(3)

for i, row in top_results.iterrows():
    print("🔹 Product Name:", row['product_name'])
    print("🔹 Brand:", row['brand'])
    print("🔹 Effects:", ", ".join(clean_text(row['notable_effects'])))
    print("🔹 Score:", row['score'])
    print("🔹 Image URL:", row['picture_src'])
    print("-"*40)


Select Skin Type:
0 : combination
1 : dry
2 : normal
3 : oily
4 : sensitive
Enter number: 0

Select Skin Condition:
0 : acne
1 : dry and dehydrated skin
2 : dull skin
3 : enlarged pores
4 : impaired skin barrier
5 : oily skin
6 : pigmentation
7 : redness
8 : skin imbalance
9 : sun damage
10 : sun protection
11 : wrinkles
Enter number: 2

Select Effect:
0 : acne-free
1 : acne-spot
2 : anti-aging
3 : balancing
4 : black-spot
5 : brightening
6 : hydrating
7 : moisturizing
8 : no-whitecast
9 : oil-control
10 : pore-care
11 : refreshing
12 : skin-barrier
13 : soothing
14 : uv-protection
Enter number: 5

✅ Predicted Product Type: Face Wash

🔥 Top Recommended Products:

🔹 Product Name: KIEHLS Ultra Facial Cleanser 150ml
🔹 Brand: KIEHL'S
🔹 Effects: brightening, anti-aging
🔹 Score: 4
🔹 Image URL: https://www.beautyhaul.com/assets/uploads/products/thumbs/800x800/635176497.g_400-w_g_.jpg
----------------------------------------
🔹 Product Name: INNERTRUE Awakening Cleansing Gel
🔹 Brand: INNERTRUE